<a href="https://colab.research.google.com/github/SandeepTripathy360/gittutorials/blob/main/cats-v-dogs-manual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install kagglehub

In [5]:
import kagglehub

kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [6]:
path = kagglehub.dataset_download(
    "shaunthesheep/microsoft-catsvsdogs-dataset"
)

print("Path:", path)

Using Colab cache for faster access to the 'microsoft-catsvsdogs-dataset' dataset.
Path: /kaggle/input/microsoft-catsvsdogs-dataset


In [7]:
import os

os.listdir(path)

['PetImages', 'readme[1].txt', 'MSR-LA - 3467.docx']

In [8]:
train_dir = path + "/PetImages"

In [9]:
os.listdir(train_dir)

['Dog', 'Cat']

Now that we have the path to our training data, let's import the necessary libraries and prepare the data for our model.

In [10]:
import tensorflow as tf
from tensorflow import keras
from keras import layers

# Define image size and batch size
image_size = (256, 256)
batch_size = 32

We will use `image_dataset_from_directory` to load the images. This utility automatically infers the class labels from the directory names (`Cat` and `Dog`).

In [7]:
import kagglehub
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from keras import layers
import os

# Re-initialize path and train_dir in case of kernel restart or out-of-order execution
path = kagglehub.dataset_download(
    "shaunthesheep/microsoft-catsvsdogs-dataset"
)
print("Path:", path)
train_dir = path + "/PetImages"

# Define image size and batch size
image_size = (256, 256)
batch_size = 32

# Get all file paths and their labels
image_paths = []
labels = []

class_names = sorted([d.name for d in os.scandir(train_dir) if d.is_dir()])
class_to_label = {class_name: i for i, class_name in enumerate(class_names)}

for class_name in class_names:
    class_path = os.path.join(train_dir, class_name)
    for img_name in os.listdir(class_path):
        if img_name.lower() == 'thumbs.db':  # Filter out Thumbs.db files
            continue
        img_path = os.path.join(class_path, img_name)
        image_paths.append(img_path)
        labels.append(class_to_label[class_name])

image_paths = np.array(image_paths)
labels = np.array(labels)

# Split data into training and validation sets
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.2, random_state=1337, stratify=labels
)

# Define a function to load, decode, resize, and normalize images
def load_and_preprocess_image(image_path, label):
    # Wrapper function for tf.py_function to handle errors
    def _preprocess_py_func(img_path_str, lbl_int):
        try:
            img_data = tf.io.read_file(img_path_str)
            img_tensor = tf.image.decode_image(img_data, channels=3, expand_animations=False)

            # Check if the image decoded properly (e.g., has 3 dimensions)
            # tf.image.decode_image for corrupt files might return a scalar or non-image shape
            if img_tensor.shape.rank != 3:
                # If decoding yields a non-3D tensor (e.g., scalar), consider it corrupt
                # Return a black image and a special label (-1)
                return np.zeros((*image_size, 3), dtype=np.float32), np.array(-1, dtype=np.int32)

            img_tensor = tf.image.resize(img_tensor, image_size)
            img_tensor = tf.cast(img_tensor, tf.float32) / 255.0
            return img_tensor.numpy(), lbl_int.numpy()
        except tf.errors.InvalidArgumentError:
            # Handle explicit decoding errors (e.g., unsupported format or corrupt header)
            return np.zeros((*image_size, 3), dtype=np.float32), np.array(-1, dtype=np.int32)
        except Exception as e:
            # Catch any other unexpected errors during processing
            print(f"Error processing image {img_path_str}: {e}") # This print will work with tf.py_function
            return np.zeros((*image_size, 3), dtype=np.float32), np.array(-1, dtype=np.int32)

    # Use tf.py_function to wrap the Python logic for error handling
    processed_image, processed_label = tf.py_function(
        func=_preprocess_py_func,
        inp=[image_path, label],
        Tout=[tf.float32, tf.int32]
    )

    # Set the static shape for the output tensors
    processed_image.set_shape((*image_size, 3))
    processed_label.set_shape([]) # Scalar label

    return processed_image, processed_label

# Create TensorFlow datasets
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))

# Map the preprocessing function to the datasets
train_ds = train_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

# Filter out the bad images (where label is -1)
train_ds = train_ds.filter(lambda image, label: tf.not_equal(label, -1))
val_ds = val_ds.filter(lambda image, label: tf.not_equal(label, -1))

# Batch, shuffle, cache, and prefetch for performance
train_ds = train_ds.cache().shuffle(buffer_size=1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)

print(f"Found {len(train_paths)} files for training.")
print(f"Found {len(val_paths)} files for validation.")

Using Colab cache for faster access to the 'microsoft-catsvsdogs-dataset' dataset.
Path: /kaggle/input/microsoft-catsvsdogs-dataset
Found 20000 files for training.
Found 5000 files for validation.


In [19]:
# The preprocessing (including normalization) is now handled directly in the dataset loading step (f4ca5c8f).
# This cell is no longer needed as the datasets are already processed.

In [14]:
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten
from keras import Sequential

In [15]:
#create CNN model
model = Sequential()

model.add(Conv2D(32, kernel_size=(3,3),padding='valid',activation='relu',input_shape=(256,256,3)))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(64, kernel_size=(3,3),padding='valid',activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(128, kernel_size=(3,3),padding='valid',activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dense(64,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [2]:
# Re-defining the CNN model as it was lost from memory
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten
from keras import Sequential

model = Sequential()

model.add(Conv2D(32, kernel_size=(3,3),padding='valid',activation='relu',input_shape=(256,256,3)))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(64, kernel_size=(3,3),padding='valid',activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(128, kernel_size=(3,3),padding='valid',activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dense(64,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [3]:
# Re-compiling the model
model.compile(optimizer='Adam', loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
# Now we can re-attempt to fit the model
history=model.fit(train_ds,epochs=10,validation_data=val_ds)

Epoch 1/10
    351/Unknown 114s 275ms/step - accuracy: 0.5764 - loss: 0.7388

In [16]:
model.compile(optimizer='Adam', loss='binary_crossentropy',metrics=['accuracy'])


In [1]:
history=model.fit(train_ds,epochs=10,validation_data=val_ds)

NameError: name 'model' is not defined